In [1]:
from pathlib import Path
import pandas as pd
import pickle
import torch
import re
from collections import Counter

PROJECT_ROOT = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())

class Vocab:
    def __init__(self, token_lists, min_freq=5):
        counts = Counter(t for toks in token_lists for t in toks)
        self.itos = [PAD, SOS, EOS, UNK] + sorted(w for w, c in counts.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, toks):
        return [self.stoi[SOS]] + [self.stoi.get(t, self.stoi[UNK]) for t in toks] + [self.stoi[EOS]]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i not in (0, 1, 2))

train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv")
val_df   = pd.read_csv(ARTIFACTS_DIR / "val.csv")
test_df  = pd.read_csv(ARTIFACTS_DIR / "test.csv")

with open(ARTIFACTS_DIR / "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open(ARTIFACTS_DIR / "features.pkl", "rb") as f:
    feat_data = pickle.load(f)

filenames_list = feat_data["filenames"]
feature_matrix = feat_data["matrix"]              # (8091, 2048)
fname_to_idx = {f: i for i, f in enumerate(filenames_list)}   # filename -> row index

print(train_df.shape, val_df.shape, test_df.shape, len(vocab.itos), feature_matrix.shape)

(32360, 2) (4045, 2) (4050, 2) 2652 torch.Size([8091, 2048])


In [2]:
from torch.utils.data import Dataset

class CaptionDataset(Dataset):
    def __init__(self, df, feature_matrix, fname_to_idx, vocab):
        self.df = df.reset_index(drop=True)
        self.feature_matrix = feature_matrix
        self.fname_to_idx = fname_to_idx
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image"]
        caption = row["caption"]

        feat_idx = self.fname_to_idx[fname]
        feature = self.feature_matrix[feat_idx].float()   # back to float32 for training

        tokens = tokenize(caption)
        ids = self.vocab.encode(tokens)                   # [<sos>, w1, w2, ..., <eos>]

        return feature, torch.tensor(ids, dtype=torch.long)

train_dataset = CaptionDataset(train_df, feature_matrix, fname_to_idx, vocab)
val_dataset   = CaptionDataset(val_df, feature_matrix, fname_to_idx, vocab)
test_dataset  = CaptionDataset(test_df, feature_matrix, fname_to_idx, vocab)

print(len(train_dataset), len(val_dataset), len(test_dataset))

feat, ids = train_dataset[0]
print(feat.shape, feat.dtype)
print(ids.shape, ids)
print(vocab.decode(ids.tolist()))

32360 4045 4050
torch.Size([2048]) torch.float32
torch.Size([19]) tensor([   1,    4,  447, 1131,    4, 1652,  672, 1161,  480, 2507,    4, 1951,
        1492, 2178, 1131,   47,    3, 2577,    2])
a child in a pink dress is climbing up a set of stairs in an <unk> way


In [3]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    features, captions = zip(*batch)

    features = torch.stack(features)                      # (batch_size, 2048)

    lengths = torch.tensor([len(c) for c in captions])
    captions_padded = pad_sequence(captions, batch_first=True, padding_value=vocab.stoi[PAD])

    return features, captions_padded, lengths

from torch.utils.data import DataLoader

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

feat_batch, cap_batch, len_batch = next(iter(train_loader))
print(feat_batch.shape)
print(cap_batch.shape)
print(len_batch[:5])
print(cap_batch[0])

torch.Size([32, 2048])
torch.Size([32, 24])
tensor([18, 15, 22, 10, 12])
tensor([   1,    4, 2648,  285, 2579,    4,  241,  153, 2260,  833, 2375, 2358,
          28,    6,    4, 2295, 1704,    2,    0,    0,    0,    0,    0,    0])


In [4]:
import torch.nn as nn

class DecoderLSTM(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=1):
        super().__init__()
        self.init_h = nn.Linear(feature_dim, hidden_dim)   
        self.init_c = nn.Linear(feature_dim, hidden_dim)   

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions):
        h0 = self.init_h(features).unsqueeze(0)  
        c0 = self.init_c(features).unsqueeze(0)

        embedded = self.embedding(captions)        
        lstm_out, _ = self.lstm(embedded, (h0, c0)) 

        logits = self.fc_out(lstm_out)              
        return logits

In [5]:
model = DecoderLSTM(feature_dim=2048, embed_dim=256, hidden_dim=512, vocab_size=len(vocab.itos))
logits = model(feat_batch, cap_batch)
print(logits.shape)

torch.Size([32, 24, 2652])


In [6]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[PAD])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_batch(features, captions):
    features = features.to(device)
    captions = captions.to(device)

    inputs = captions[:, :-1]    
    targets = captions[:, 1:]   

    optimizer.zero_grad()
    logits = model(features, inputs)             

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),      
        targets.reshape(-1)                        
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()

    return loss.item()

loss = train_one_batch(feat_batch, cap_batch)
print(loss)

7.885123252868652


In [7]:
from tqdm import tqdm
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for features, captions, lengths in loader:
            features = features.to(device)
            captions = captions.to(device)

            inputs = captions[:, :-1]
            targets = captions[:, 1:]

            logits = model(features, inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

            total_loss += loss.item()
            n_batches += 1

    model.train()
    return total_loss / n_batches


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for features, captions, lengths in tqdm(loader):
        loss = train_one_batch(features, captions)
        total_loss += loss
        n_batches += 1

    return total_loss / n_batches


NUM_EPOCHS = 10
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate(val_loader)
    print(f"epoch {epoch+1}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), ARTIFACTS_DIR / "best_model.pt")
        print(f"  -> saved new best model (val_loss={val_loss:.4f})")

100%|██████████| 1012/1012 [00:18<00:00, 54.71it/s]


epoch 1/10  train_loss=3.3660  val_loss=2.8806
  -> saved new best model (val_loss=2.8806)


100%|██████████| 1012/1012 [00:16<00:00, 59.79it/s]


epoch 2/10  train_loss=2.5839  val_loss=2.7044
  -> saved new best model (val_loss=2.7044)


100%|██████████| 1012/1012 [00:17<00:00, 59.13it/s]


...

epoch 8/10  train_loss=1.1765  val_loss=3.0290


100%|██████████| 1012/1012 [00:17<00:00, 57.95it/s]


epoch 9/10  train_loss=1.0252  val_loss=3.1550


100%|██████████| 1012/1012 [00:17<00:00, 57.62it/s]


epoch 10/10  train_loss=0.8930  val_loss=3.2807


In [10]:
import os
path = ARTIFACTS_DIR / "best_model.pt"
print(path.exists(), os.path.getsize(path) / 1024, "KB")

True 22326.2412109375 KB


In [11]:
inference_model = DecoderLSTM(feature_dim=2048, embed_dim=256, hidden_dim=512, vocab_size=len(vocab.itos))
inference_model.load_state_dict(torch.load(ARTIFACTS_DIR / "best_model.pt"))
inference_model = inference_model.to(device)
inference_model.eval()

print("loaded")

loaded


In [13]:
@torch.no_grad()
def generate_caption(feature, model, vocab, max_len=25):
    model.eval()
    feature = feature.unsqueeze(0).to(device)   # (1, 2048) -- batch of one

    h = model.init_h(feature).unsqueeze(0)      # (1, 1, hidden_dim)
    c = model.init_c(feature).unsqueeze(0)

    current_token = torch.tensor([[vocab.stoi[SOS]]], device=device)  # start with <sos>
    generated_ids = []

    for _ in range(max_len):
        embedded = model.embedding(current_token)          # (1, 1, embed_dim)
        lstm_out, (h, c) = model.lstm(embedded, (h, c))     # one step, carrying memory forward
        logits = model.fc_out(lstm_out.squeeze(1))          # (1, vocab_size)

        next_id = logits.argmax(dim=-1).item()              # greedy: pick the single best word

        if next_id == vocab.stoi[EOS]:
            break

        generated_ids.append(next_id)
        current_token = torch.tensor([[next_id]], device=device)   # feed prediction back in as next input

    return vocab.decode(generated_ids)


# try it on a test-set image
test_feat_idx = fname_to_idx[test_df.iloc[0]["image"]]
test_feature = feature_matrix[test_feat_idx].float()

import random

sample_idxs = random.sample(range(len(test_df)), 5)

for idx in sample_idxs:
    row = test_df.iloc[idx]
    feat_idx = fname_to_idx[row["image"]]
    feature = feature_matrix[feat_idx].float()

    generated = generate_caption(feature, inference_model, vocab)
    print(f"Image: {row['image']}")
    print(f"  Generated:    {generated}")
    print(f"  Ground truth: {row['caption']}")
    print()

Image: 3173157541_70b16b4318.jpg
  Generated:    a man in a red hat and a hat
  Ground truth: A man in a white hat is standing behind a woman with a red shawl over her head .

Image: 856985136_649c0a3881.jpg
  Generated:    a girl in a blue shirt is jumping into a pool
  Ground truth: Woman balancing on edge of fountain while sticking her toe in the water .

Image: 2744330402_824240184c.jpg
  Generated:    a group of people are sitting in a boat in the middle of a field
  Ground truth: A large group of people and possessions packed into a small boat .

Image: 3259002340_707ce96858.jpg
  Generated:    a black dog with a blue collar is running with a black dog
  Ground truth: The black dog is wearing a purple collar with a black leash as he is running .

Image: 3677693858_62f2f3163f.jpg
  Generated:    a young boy in a red shirt is jumping on a swing
  Ground truth: A child jumping out of a swing near some bushes .



In [14]:
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

In [15]:
def compute_bleu_for_test_set(test_df, feature_matrix, fname_to_idx, model, vocab):
    # group ground-truth captions by image (each image has 5 references)
    refs_by_image = test_df.groupby("image")["caption"].apply(list).to_dict()

    bleu1_scores, bleu4_scores = [], []

    for fname, references in refs_by_image.items():
        feat_idx = fname_to_idx[fname]
        feature = feature_matrix[feat_idx].float()

        generated = generate_caption(feature, model, vocab)
        gen_tokens = generated.split()

        ref_tokens_list = [tokenize(ref) for ref in references]   # 5 tokenized references

        bleu1 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)
        bleu4 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

        bleu1_scores.append(bleu1)
        bleu4_scores.append(bleu4)

    return sum(bleu1_scores) / len(bleu1_scores), sum(bleu4_scores) / len(bleu4_scores)


avg_bleu1, avg_bleu4 = compute_bleu_for_test_set(test_df, feature_matrix, fname_to_idx, inference_model, vocab)
print(f"BLEU-1: {avg_bleu1:.4f}")
print(f"BLEU-4: {avg_bleu4:.4f}")

BLEU-1: 0.5693
BLEU-4: 0.1709
